In [1]:
"""
Real SHARAD Orbit 571601 Processing

Full production pipeline:
1. Load orbit 571601 data
2. Generate cluttergram
3. Compare with paper Figure 2
4. Generate echo map
5. Analyze results
"""

import numpy as np
import matplotlib.pyplot as plt
import sys
from pathlib import Path

sys.path.append('../src')

from sharad_reader import SHARADReader, visualize_orbit_geometry
from production_pipeline import (
    run_full_simulation,
    compare_cluttergram_radargram,
    generate_echo_map
)

print("="*70)
print("REAL SHARAD ORBIT 571601 PROCESSING")
print("="*70)

# ============================================================================
# STEP 1: LOAD AND VISUALIZE ORBIT GEOMETRY
# ============================================================================

print("\n" + "="*70)
print("STEP 1: LOAD SHARAD ORBIT 571601")
print("="*70)

# Path to SHARAD data
sharad_base = Path('../data/sharad/s_00571601_rgram')
mola_file = Path('../data/mola/megt00n090hb.img')
output_dir = Path('../data/outputs/orbit_571601')

print(f"\nLoading SHARAD data from: {sharad_base}")

reader = SHARADReader(sharad_base)

# Get orbit info
orbit_info = reader.get_orbit_info()

print(f"\nOrbit 571601 Information:")
print(f"  Records: {orbit_info['n_records']}")
print(f"  Latitude range: {orbit_info['lat_range'][0]:.2f}° to {orbit_info['lat_range'][1]:.2f}°")
print(f"  Longitude range: {orbit_info['lon_range'][0]:.2f}° to {orbit_info['lon_range'][1]:.2f}°")
print(f"  Mean altitude: {orbit_info['altitude_mean']/1000:.1f} km")

# Visualize geometry
print("\nVisualizing orbit geometry...")
fig_geom = visualize_orbit_geometry(reader)
plt.savefig(output_dir / 'orbit_571601_geometry.png', dpi=150, bbox_inches='tight')
print("  ✓ Saved: orbit_571601_geometry.png")
plt.show()

# ============================================================================
# STEP 2: RUN SIMULATION (SUBSAMPLED FOR TESTING)
# ============================================================================

print("\n" + "="*70)
print("STEP 2: RUN CLUTTER SIMULATION (SUBSAMPLED)")
print("="*70)

# For testing, subsample by factor 10 (faster)
# For production, use subsample_factor=1
subsample_factor = 10

print(f"\nRunning simulation with subsample factor {subsample_factor}...")
print(f"  This will process {orbit_info['n_records']//subsample_factor} positions")
print(f"  (Full orbit has {orbit_info['n_records']} positions)")

results = run_full_simulation(
    sharad_base_path=sharad_base,
    mola_file=mola_file,
    output_dir=output_dir,
    subsample_factor=subsample_factor,
    cross_track_extent=45000,
    facet_size_cross=30.0,
    facet_size_along=300.0,
    verbose=True
)

print("\n✓ Simulation complete!")

# ============================================================================
# STEP 3: COMPARE WITH REAL RADARGRAM
# ============================================================================

print("\n" + "="*70)
print("STEP 3: COMPARE SIMULATED VS REAL")
print("="*70)

cluttergram = results['cluttergram']
radargram = results['radargram']

print(f"\nCluttergram shape: {cluttergram.data.shape}")
print(f"Radargram shape: {radargram.shape}")

# Create comparison plot
fig_comparison = compare_cluttergram_radargram(
    cluttergram, 
    radargram,
    output_file=output_dir / 'comparison_simulated_vs_real.png'
)
plt.show()

# ============================================================================
# STEP 4: DETAILED ANALYSIS
# ============================================================================

print("\n" + "="*70)
print("STEP 4: DETAILED ANALYSIS")
print("="*70)

# Analyze cluttergram structure
print("\nCluttergram statistics:")
nonzero_bins = np.sum(cluttergram.data > 0)
total_bins = cluttergram.data.size
fill_rate = nonzero_bins / total_bins

print(f"  Non-zero bins: {nonzero_bins:,} / {total_bins:,}")
print(f"  Fill rate: {fill_rate*100:.2f}%")
print(f"  Max power: {cluttergram.data.max():.2e}")
print(f"  Mean power (non-zero): {cluttergram.data[cluttergram.data>0].mean():.2e}")

# Power distribution analysis
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Along-track power profile
ax1 = axes[0, 0]
power_per_pos = cluttergram.data.sum(axis=1)
ax1.plot(power_per_pos, 'b-', linewidth=1)
ax1.set_xlabel('Position Index')
ax1.set_ylabel('Total Power')
ax1.set_title('Cluttergram: Power vs Position', fontweight='bold')
ax1.grid(True, alpha=0.3)

# Compare with radargram
radargram_power_per_pos = np.sum(np.abs(radargram), axis=1)
ax1_twin = ax1.twinx()
ax1_twin.plot(radargram_power_per_pos, 'r-', linewidth=1, alpha=0.5)
ax1_twin.set_ylabel('Radargram Power', color='r')

# Time-delay power profile
ax2 = axes[0, 1]
power_per_bin = cluttergram.data.sum(axis=0)
ax2.plot(power_per_bin, 'g-', linewidth=1)
ax2.set_xlabel('Time Bin')
ax2.set_ylabel('Total Power')
ax2.set_title('Cluttergram: Power vs Time Delay', fontweight='bold')
ax2.grid(True, alpha=0.3)
ax2.set_yscale('log')

# 2D histogram
ax3 = axes[1, 0]
im3 = ax3.imshow(np.log10(cluttergram.data.T + 1), 
                cmap='viridis', aspect='auto', origin='upper')
ax3.set_xlabel('Position')
ax3.set_ylabel('Time Bin')
ax3.set_title('Cluttergram: log₁₀(Power)', fontweight='bold')
plt.colorbar(im3, ax=ax3, fraction=0.046)

# Cross-correlation between simulated and real
ax4 = axes[1, 1]

# Normalize both for comparison
sim_norm = power_per_pos / power_per_pos.max() if power_per_pos.max() > 0 else power_per_pos
real_norm = radargram_power_per_pos / radargram_power_per_pos.max()

ax4.plot(sim_norm, 'b-', linewidth=2, label='Simulated', alpha=0.7)
ax4.plot(real_norm, 'r-', linewidth=2, label='Real', alpha=0.7)
ax4.set_xlabel('Position Index')
ax4.set_ylabel('Normalized Power')
ax4.set_title('Power Profile Comparison', fontweight='bold')
ax4.legend()
ax4.grid(True, alpha=0.3)

# Compute correlation
correlation = np.corrcoef(sim_norm, real_norm)[0, 1]
ax4.text(0.05, 0.95, f'Correlation: {correlation:.3f}',
        transform=ax4.transAxes, fontsize=12,
        verticalalignment='top',
        bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig(output_dir / 'detailed_analysis.png', dpi=150, bbox_inches='tight')
print("\n✓ Saved: detailed_analysis.png")
plt.show()

print(f"\nSimulated vs Real correlation: {correlation:.3f}")

# ============================================================================
# STEP 5: GENERATE ECHO MAP
# ============================================================================

print("\n" + "="*70)
print("STEP 5: GENERATE ECHO MAP")
print("="*70)

fig_echo = generate_echo_map(
    cluttergram,
    results['spacecraft_positions'],
    results['nadir_positions'],
    output_file=output_dir / 'echo_map.png'
)
plt.show()

# ============================================================================
# STEP 6: PAPER COMPARISON
# ============================================================================

print("\n" + "="*70)
print("STEP 6: COMPARISON WITH PAPER FIGURE 2")
print("="*70)

print("\nPaper (Choudhary et al. 2016) Figure 2 shows:")
print("  - Eastern Hellas region")
print("  - Orbit 571601")
print("  - Strong clutter returns from rough terrain")
print("  - Clutter extends to ~100 μs delay")

# Check our results match
time_extent = np.where(power_per_bin > 0)[0]
if len(time_extent) > 0:
    max_time_bin = time_extent[-1]
    max_time_us = max_time_bin * cluttergram.time_bin_width * 1e6
    
    print(f"\nOur simulation:")
    print(f"  Orbit: {orbit_info['orbit_number']}")
    print(f"  Region: Lat {orbit_info['lat_range']}, Lon {orbit_info['lon_range']}")
    print(f"  Max time delay: {max_time_us:.1f} μs")
    print(f"  (Paper shows ~100-120 μs)")

# Create figure similar to paper's Figure 2
fig_paper_style = plt.figure(figsize=(16, 12))

# Top: Ground track with elevation
ax1 = plt.subplot(3, 1, 1)
lats, lons = reader.get_ground_track()
ground_elev = []
for lat, lon in zip(lats[::subsample_factor], lons[::subsample_factor]):
    ground_elev.append(results['interpolator'](lat, lon))
ground_elev = np.array(ground_elev)

ax1_twin = ax1.twinx()
ax1.plot(lats[::subsample_factor], 'b-', linewidth=1, label='Latitude')
ax1_twin.plot(ground_elev, 'g-', linewidth=1, label='Elevation')
ax1.set_ylabel('Latitude (°N)', color='b', fontsize=12)
ax1_twin.set_ylabel('Elevation (m)', color='g', fontsize=12)
ax1.set_title(f'SHARAD Orbit {orbit_info["orbit_number"]} - Ground Track',
             fontsize=14, fontweight='bold')
ax1.grid(True, alpha=0.3)

# Middle: Cluttergram
ax2 = plt.subplot(3, 1, 2)
clutter_norm = cluttergram.normalize(method='dB', db_range=80)
im2 = ax2.imshow(clutter_norm.T, cmap='gray', aspect='auto', origin='upper')
ax2.set_ylabel('Time Delay (μs)', fontsize=12)
ax2.set_title('Simulated Cluttergram', fontsize=14, fontweight='bold')

# Fix Y-axis labels to show microseconds
n_bins = cluttergram.n_time_bins
max_time_us = cluttergram.max_time_delay * 1e6
y_ticks = ax2.get_yticks()
y_labels = [f'{int(tick * max_time_us / n_bins)}' for tick in y_ticks]
ax2.set_yticklabels(y_labels)

plt.colorbar(im2, ax=ax2, fraction=0.046, label='Power (dB)')

# Bottom: Real radargram
ax3 = plt.subplot(3, 1, 3)
radargram_safe = np.abs(radargram) + 1e-30
radargram_db = 10 * np.log10(radargram_safe)
radargram_db_max = np.max(radargram_db)
radargram_db_min = radargram_db_max - 80
radargram_db_clip = np.clip(radargram_db, radargram_db_min, radargram_db_max)
radargram_norm = ((radargram_db_clip - radargram_db_min) / 80 * 255).astype(np.uint8)

im3 = ax3.imshow(radargram_norm.T, cmap='gray', aspect='auto', origin='upper')
ax3.set_xlabel('Along-track Position', fontsize=12)
ax3.set_ylabel('Sample Number', fontsize=12)
ax3.set_title('Real SHARAD Radargram', fontsize=14, fontweight='bold')
plt.colorbar(im3, ax=ax3, fraction=0.046, label='Power (dB)')

plt.tight_layout()
plt.savefig(output_dir / 'paper_style_comparison.png', dpi=150, bbox_inches='tight')
print("\n✓ Saved: paper_style_comparison.png")
plt.show()

# ============================================================================
# SUMMARY
# ============================================================================

print("\n" + "="*70)
print("SUMMARY")
print("="*70)

print(f"\n✓ Successfully processed SHARAD orbit {orbit_info['orbit_number']}")
print(f"\nResults:")
print(f"  Positions processed: {len(results['spacecraft_positions'])}")
print(f"  Cluttergram shape: {cluttergram.data.shape}")
print(f"  Fill rate: {fill_rate*100:.2f}%")
print(f"  Correlation with real data: {correlation:.3f}")
print(f"  Max time delay: {max_time_us:.1f} μs")

print(f"\nOutput files saved to: {output_dir}")
print(f"  - orbit_571601_geometry.png")
print(f"  - comparison_simulated_vs_real.png")
print(f"  - detailed_analysis.png")
print(f"  - echo_map.png")
print(f"  - paper_style_comparison.png")
print(f"  - cluttergram_orbit_571601.npz")

print("\n" + "="*70)
print("🎉 PRODUCTION PIPELINE COMPLETE!")
print("="*70)

print("\nNext steps:")
print("  • Run with subsample_factor=1 for full resolution")
print("  • Compare quantitatively with paper metrics")
print("  • Generate left/right clutter separation")
print("  • Process additional orbits")
print("  • Implement automated clutter flagging")

Configuration loaded:
  Wavelength: 14.99 m
  Range resolution: 5.62 m
  Facets per position: 3000
  Time bins: 3600
REAL SHARAD ORBIT 571601 PROCESSING

STEP 1: LOAD SHARAD ORBIT 571601

Loading SHARAD data from: ..\data\sharad\s_00571601_rgram
SHARAD Reader initialized:
  IMG: s_00571601_rgram.img
  XML: s_00571601_rgram.xml

Parsing XML geometry file...


ValueError: Cannot find ancillary_data in XML